# v64d v5 — KTF³-R Spin Alignment Test (Galaxy Zoo 1)

**Pre-registered:** osf.io/42nks — 4 April 2026 00:36 UTC  
**Prediction:** CCW spin preference within 30° of KTF³ axis  
**Key:** North pole l=97°, b=+31° → Dec≈+16° is VISIBLE to SDSS/GZ1!  
**Fix:** GZ1 uses sexagesimal RA (HH:MM:SS) and DEC (DD:MM:SS) format.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from astropy.coordinates import SkyCoord
from astropy.coordinates import Angle
import astropy.units as u
import os, gzip, shutil

CSV = 'GalaxyZoo1_DR_table2.csv'
GZ  = 'GalaxyZoo1_DR_table2.csv.gz'

if not os.path.exists(CSV) or os.path.getsize(CSV) < 100000:
    if os.path.exists(GZ):
        print('Decompressing...')
        with gzip.open(GZ, 'rb') as fi, open(CSV, 'wb') as fo:
            shutil.copyfileobj(fi, fo)
        print('Done:', round(os.path.getsize(CSV)/1e6), 'MB')
    else:
        raise FileNotFoundError('Upload GalaxyZoo1_DR_table2.csv.gz')
else:
    print('Ready:', round(os.path.getsize(CSV)/1e6), 'MB')


In [ ]:
# Load CSV
df = pd.read_csv(CSV, sep=',', skipinitialspace=True)
df.columns = [c.strip() for c in df.columns]
print('Rows:', len(df))
print('Columns:', list(df.columns))
print()
print('First 2 rows:')
print(df.head(2).to_string())


In [ ]:
# === CONVERT SEXAGESIMAL RA/DEC TO DECIMAL DEGREES ===
# GZ1 uses format HH:MM:SS.ss for RA and DD:MM:SS.s for DEC

print('Converting RA from HH:MM:SS to decimal degrees...')
ra_deg = Angle(df['RA'].str.strip().tolist(), unit=u.hourangle).deg
print('RA range:', round(float(ra_deg.min()), 1), '-', round(float(ra_deg.max()), 1))

print('Converting DEC from DD:MM:SS to decimal degrees...')
dec_deg = Angle(df['DEC'].str.strip().tolist(), unit=u.deg).deg
print('DEC range:', round(float(dec_deg.min()), 1), '-', round(float(dec_deg.max()), 1))

# Add as new columns
df['ra_deg']  = ra_deg
df['dec_deg'] = dec_deg

# Convert spin columns
df['P_CW']  = pd.to_numeric(df['P_CW'],  errors='coerce')
df['P_ACW'] = pd.to_numeric(df['P_ACW'], errors='coerce')

# Drop rows with missing coordinates
df = df.dropna(subset=['ra_deg', 'dec_deg']).reset_index(drop=True)
print()
print('Galaxies after cleaning:', len(df))
print('P_CW  mean:', round(float(df['P_CW'].mean()), 4))
print('P_ACW mean:', round(float(df['P_ACW'].mean()), 4))


In [ ]:
# === KTF³ AXIS — BOTH POLES ===
south = SkyCoord(l=277.0*u.deg, b=-31.0*u.deg, frame='galactic')
north = SkyCoord(l=97.0*u.deg,  b=+31.0*u.deg, frame='galactic')
ra_s,  dec_s  = float(south.icrs.ra.deg), float(south.icrs.dec.deg)
ra_n,  dec_n  = float(north.icrs.ra.deg), float(north.icrs.dec.deg)

print('South pole: RA =', round(ra_s, 1), '  Dec =', round(dec_s, 1), '  (outside SDSS)')
print('North pole: RA =', round(ra_n, 1), '  Dec =', round(dec_n, 1), '  (SDSS visible!)')

ra  = df['ra_deg'].values
dec = df['dec_deg'].values

def angular_sep(ra1, dec1, ra2, dec2):
    c = (np.sin(np.radians(dec1))*np.sin(np.radians(dec2)) +
         np.cos(np.radians(dec1))*np.cos(np.radians(dec2))*np.cos(np.radians(ra1-ra2)))
    return np.degrees(np.arccos(np.clip(c, -1, 1)))

sep_s = angular_sep(ra, dec, ra_s, dec_s)
sep_n = angular_sep(ra, dec, ra_n, dec_n)

NEAR, FAR = 30.0, 60.0
print()
print('Near south pole (<30 deg):', int((sep_s < NEAR).sum()))
print('Near north pole (<30 deg):', int((sep_n < NEAR).sum()), '  <-- main test')
print('Far from north  (>60 deg):', int((sep_n > FAR).sum()))


In [ ]:
# === SPIN TEST ===
p_cw  = df['P_CW'].values
p_acw = df['P_ACW'].values
total = p_cw + p_acw

# Select spirals with clear spin vote
ok = np.isfinite(total) & (total > 0.3) & np.isfinite(p_cw) & np.isfinite(p_acw)
frac_cw = np.where(ok, p_cw / np.where(total > 0, total, np.nan), np.nan)

print('Galaxies with clear spin vote (total > 0.3):', int(ok.sum()))
print('Overall mean p_cw:', round(float(np.nanmean(frac_cw)), 5), '  (expect ~0.5)')
print()

# Test both poles
for label, near_mask in [
    ('NORTH POLE  l=97, b=+31  (SDSS visible!)', sep_n < NEAR),
    ('SOUTH POLE  l=277, b=-31 (outside SDSS)',  sep_s < NEAR),
]:
    mask_near = near_mask & ok
    mask_far  = (sep_n > FAR) & ok

    print('===', label, '===')
    print('Near axis with spin vote:', int(mask_near.sum()))

    if mask_near.sum() < 5:
        print('Too few galaxies — cannot test')
        print()
        continue

    sn = frac_cw[mask_near]
    sf = frac_cw[mask_far]
    sn = sn[np.isfinite(sn)]
    sf = sf[np.isfinite(sf)]

    t_nf, _ = stats.ttest_ind(sn, sf)
    t_iso, _ = stats.ttest_1samp(sn, 0.5)

    print('Near mean p_cw :', round(float(sn.mean()), 5))
    print('Far  mean p_cw :', round(float(sf.mean()), 5))
    print('sigma (near vs far) :', round(float(t_nf),  3))
    print('sigma (near vs 0.5) :', round(float(t_iso), 3))

    if abs(t_nf) > 2:
        verdict = 'SIGNAL'
    elif abs(t_nf) > 1:
        verdict = 'MARGINAL'
    else:
        verdict = 'NULL'
    print('VERDICT:', verdict)

    if sn.mean() < 0.5:
        print('Direction: CCW preferred  (matches KTF3-R prediction!)')
    else:
        print('Direction: CW  preferred  (opposite to prediction)')
    print()


In [ ]:
# === PROFILE PLOT ===
bins = np.arange(0, 181, 15)
bc   = 0.5*(bins[:-1] + bins[1:])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('v64d v5  --  KTF3-R Spin Test  (Galaxy Zoo 1)\n'
             'Pre-registered osf.io/42nks  4 April 2026',
             fontweight='bold')

for ax, pole_sep, title, color in [
    (axes[0], sep_s, 'South pole  l=277, b=-31  (outside SDSS)', 'red'),
    (axes[1], sep_n, 'North pole  l=97,  b=+31  (SDSS visible!)', 'blue'),
]:
    means, errs = [], []
    for i in range(len(bins)-1):
        m = (pole_sep >= bins[i]) & (pole_sep < bins[i+1]) & ok
        vals = frac_cw[m]
        vals = vals[np.isfinite(vals)]
        if len(vals) > 10:
            means.append(float(vals.mean()))
            errs.append(float(vals.std() / np.sqrt(len(vals))))
        else:
            means.append(float('nan'))
            errs.append(float('nan'))
    ax.errorbar(bc, means, yerr=errs, fmt='o-', color=color,
                lw=2, capsize=4, markersize=6)
    ax.axhline(0.5, color='gray', ls='--', label='Isotropy 0.5')
    ax.axvline(NEAR, color='red',    ls=':', lw=2, label='Near 30 deg')
    ax.axvline(FAR,  color='orange', ls=':', lw=2, label='Far 60 deg')
    ax.set_ylim(0.46, 0.54)
    ax.set_xlabel('Angular separation [deg]')
    ax.set_ylabel('Mean p_cw  (0.5 = isotropic)')
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

# Count per bin
ax = axes[2]
cs = [((sep_s >= bins[i]) & (sep_s < bins[i+1]) & ok).sum() for i in range(len(bins)-1)]
cn = [((sep_n >= bins[i]) & (sep_n < bins[i+1]) & ok).sum() for i in range(len(bins)-1)]
ax.bar([b-3 for b in bc], cs, width=6, alpha=0.7, color='red',  label='South pole')
ax.bar([b+3 for b in bc], cn, width=6, alpha=0.7, color='blue', label='North pole')
ax.axvline(NEAR, color='black', ls=':', lw=2)
ax.set_xlabel('Sep from pole [deg]')
ax.set_ylabel('Galaxies with spin vote')
ax.set_title('Galaxy count per bin')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('v64d_v5_spin.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: v64d_v5_spin.png')


In [ ]:
print('=' * 60)
print('v64d v5  SUMMARY  --  KTF3-R Spin Test  --  Galaxy Zoo 1')
print('=' * 60)
print('Pre-registration : osf.io/42nks  4 April 2026 00:36 UTC')
print('Prediction       : CCW preference within 30 deg of KTF3 axis')
print()
print('KTF3 axis poles:')
print('  South  l=277, b=-31  Dec~-67  outside SDSS  -> untestable now')
print('  North  l=97,  b=+31  Dec~+16  SDSS visible   -> tested here')
print()
print('Dataset: Galaxy Zoo 1 (Lintott et al. 2011)')
print('  Total galaxies  :', len(df))
print('  With spin vote  :', int(ok.sum()))
print('  Near north pole :', int((sep_n < NEAR).sum()))
print('  Near south pole :', int((sep_s < NEAR).sum()))
print()
print('CAVEAT: Definitive test = Euclid DR1, October 2026')
print('GitHub : github.com/Andrew-Cot/KTF3-Analyse')
print('Ref    : Lintott et al. 2011  MNRAS 410 166')
